# Starter 3 — Small open-weight VLM

> **Status: skeleton.** Structure and interfaces are in place; the model-dependent cells are stubs marked `TODO(kevin)`. Baseline CERs land here before launch.

**Task recap:** given a page image from one of four UW collections (surveyors' field notes, German Kurrent letters, ledger account books, microfilm treaty documents), emit a faithful verbatim transcription. Every model used must be open-weight and under 70B parameters. See the repo README and `docs/transcription_conventions.md`.

The shortest path from image to transcription: prompt a small open-weight vision-language model (e.g. **Qwen2.5-VL-7B-Instruct** — comfortably under the 70B cap and within a 16–24 GB GPU or Kaggle's free tier) with the page and ask for a verbatim transcription. No segmentation, no routing — the model does layout, language, and recognition implicitly.

The catch, and it's exactly what this challenge measures: VLMs *summarize and normalize by default*. They fix historical spelling, drop line-level detail, skip table cells, and hallucinate plausible text on illegible regions — all scored as errors under verbatim CER. Prompting (and possibly fine-tuning) the normalization *out* is the craft here.

## 0. Setup

In [ ]:
%pip install -q transformers accelerate qwen-vl-utils pandas

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # open-weight, 7B — well under the 70B cap

## 1. Load the model

In [ ]:
# TODO(kevin): confirm exact loading incantation + dtype for Kaggle T4/P100 free tier.
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
# import torch
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
# processor = AutoProcessor.from_pretrained(MODEL_ID)

## 2. Prompt for *faithful* transcription

The prompt does the work a traditional pipeline does with architecture. Things that matter, per the scoring rules:

- demand verbatim output: original spelling, casing, punctuation — no corrections
- one output line per visual line (whitespace itself is free, but line discipline keeps the model from paraphrasing)
- tables cell-by-cell in reading order, no markdown syntax
- forbid commentary, translation, and `[illegible]`-style editorializing

In [ ]:
SYSTEM_PROMPT = """You are a professional archival transcriber. Transcribe the text in the image exactly as written.
Rules:
- Preserve original spelling, capitalization, punctuation, and abbreviations. Never modernize or correct.
- Output one line of text per line on the page, in natural reading order.
- For tables, transcribe cell contents in reading order separated by spaces. Do not add table formatting.
- Transcribe only text that is on the page. No commentary, no translation, no descriptions of images.
- German material: transcribe in German with original umlauts and eszett."""

def transcribe_page(page_path) -> str:
    """TODO(kevin): image -> messages -> generate -> decode. Consider tiling very
    large pages; TIFF-derived scans can exceed the vision encoder's budget."""
    raise NotImplementedError

## Scoring your output

Whatever pipeline you build, score it the way the leaderboard will — verbatim CER with whitespace collapsed, macro-averaged by category. `evaluation/metric.py` in the challenge repo is the single source of truth.

In [ ]:
import sys
sys.path.insert(0, "../evaluation")  # adjust if running on Kaggle: attach the challenge repo as a dataset
from metric import page_cer, page_wer

prediction = "North between Sections 13 & 14"
reference  = "North between Sections 13 & 14"
print("CER:", page_cer(prediction, reference), " WER:", page_wer(prediction, reference))

## 3. Build a submission

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/sample")
# meta = pd.read_csv(DATA_DIR / "metadata.csv")
# sub = pd.DataFrame({
#     "page_id": meta["page_id"],
#     "text": [transcribe_page(DATA_DIR / "images" / f"{pid}.jpg") for pid in meta["page_id"]],
# })
# sub.to_csv("submission.csv", index=False)

## Ideas to beat this baseline

- **Fine-tune** the VLM on curated (image, verbatim transcript) pairs — LoRA on a 7B fits hobby hardware; the fine-tune counts as its base model under the rules
- Two-pass prompting: transcribe, then a second pass comparing output to image asking "what did you normalize?"
- Crop-and-stitch: run the VLM per detected region (borrow Kraken from Starter 2) to stop table-skipping on dense ledger pages
- Sampling: temperature 0 / greedy — creativity is exactly what you don't want
- Watch the German category: small VLMs are weakest on Kurrent; this is where the traditional pipeline may still win